Tracking Hyperparameter Tuning with MLflow

In [1]:
import mlflow

/home/itxkakar/Personal /Coding Playground/Hands-on-Machine-Learning/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
mlflow.set_experiment("Hyperparamer Tuning Experiment")

<Experiment: artifact_location=('/home/itxkakar/Personal /Coding '
 'Playground/Hands-on-Machine-Learning/mlflow-starts/mlruns/3'), creation_time=1793858614291, effective_trace_archival_retention=None, experiment_id='3', last_update_time=1793858614291, lifecycle_stage='active', name='Hyperparamer Tuning Experiment', tags={}, trace_location=None, workspace='default'>

In [3]:
import pandas as pd

iris = pd.read_csv('Iris.csv')
iris.head()

,Id,SepalLengthCm,SepalWidthCm,PetalLengthCm,PetalWidthCm,Species
0,1,5.1,3.5,1.4,0.2,Iris-setosa
1,2,4.9,3.0,1.4,0.2,Iris-setosa
2,3,4.7,3.2,1.3,0.2,Iris-setosa
3,4,4.6,3.1,1.5,0.2,Iris-setosa
4,5,5.0,3.6,1.4,0.2,Iris-setosa


In [4]:
X = iris.drop(columns=['Id', "Species"] )
y = iris['Species']
X.shape, y.shape

((150, 4), (150,))

In [5]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
X_train.shape, y_train.shape

((120, 4), (120,))

In [6]:
# TODO: fix the random forest classifier

# from sklearn.ensemble import RandomForestClassifier
# from sklearn.metrics import accuracy_score
# import optuna
# import sklearn

# # optuna.Trial.suggest_categorical


# def objective(trial):
#     with mlflow.start_run(nested=True, run_name=f"trail_{trial.number}") as child_run:

#         cf_n_estimators = trial.suggest_int("cf_n_estimators", 50, 300, step=10)
#         cf_criterion = trial.suggest_categorical("cf_criterion", ['gini', 'entropy', 'log_loss'])
#         cf_bootstrap = trial.suggest_categorical("cf_bootstrap",[False, True])

#         params = {
#             "n_estimators": cf_n_estimators,
#             "criterion": cf_criterion,
#             "bootstrap": cf_bootstrap
#         }

#         # log the current params 
#         mlflow.log_params(params)

#         cf_obj = RandomForestClassifier(**params)
#         cf_obj.fit(X_train, y_train)

#         y_pred = cf_obj.predict(X_test)
#         accuracy = accuracy_score(y_test, y_pred)

#         # log the current trial's accuracy
#         mlflow.metrics({"accuracy": accuracy})

#         # log the model file
#         mlflow.sklearn.log_model(cf_obj, name="model")

#         # Make it easy to reterive the best-performing model
#         trial.set_user_attr("run_id", child_run.info.run_id)

#         return accuracy

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
import optuna
import sklearn

In [9]:
def objective(trial):
    with mlflow.start_run(
        nested=True,
        run_name=f"trial_{trial.number}"
    ) as child_run:

        params = {
            "max_iter": trial.suggest_int("max_iter", 100, 500, step=50),
            "solver": trial.suggest_categorical(
                "solver",
                ["lbfgs", "newton-cg", "sag", "saga"]
            ),
            "C": trial.suggest_float("C", 0.01, 10.0, log=True),
        }

        mlflow.log_params(params)

        model = LogisticRegression(**params)
        model.fit(X_train, y_train)

        accuracy = accuracy_score(y_test, model.predict(X_test))

        mlflow.log_metric("accuracy", accuracy)
        mlflow.sklearn.log_model(model, name="model")

        trial.set_user_attr("run_id", child_run.info.run_id)

        return accuracy


with mlflow.start_run(run_name="study") as run:
    n_trials = 10
    mlflow.log_param("n_trials", n_trials)

    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=n_trials)

    mlflow.log_params(study.best_trial.params)
    mlflow.log_metric("best_accuracy", study.best_value)

    best_run_id = study.best_trial.user_attrs.get("run_id")
    if best_run_id:
        mlflow.log_param("best_child_run_id", best_run_id)

[I 2026-11-06 09:04:16,242] A new study created in memory with name: no-name-af47c953-c5d7-4b97-8ff0-395ecc7640ea
[I 2026-11-06 09:04:35,385] Trial 0 finished with value: 1.0 and parameters: {'max_iter': 350, 'solver': 'newton-cg', 'C': 2.626628624110469}. Best is trial 0 with value: 1.0.
[I 2026-11-06 09:04:49,389] Trial 1 finished with value: 0.9333333333333333 and parameters: {'max_iter': 350, 'solver': 'lbfgs', 'C': 0.013459847290710562}. Best is trial 0 with value: 1.0.
[I 2026-11-06 09:04:58,173] Trial 2 finished with value: 0.9666666666666667 and parameters: {'max_iter': 500, 'solver': 'newton-cg', 'C': 0.9771924270979788}. Best is trial 0 with value: 1.0.
[I 2026-11-06 09:05:12,067] Trial 3 finished with value: 0.9333333333333333 and parameters: {'max_iter': 500, 'solver': 'newton-cg', 'C': 0.017333396973685505}. Best is trial 0 with value: 1.0.
[I 2026-11-06 09:05:20,736] Trial 4 finished with value: 0.9666666666666667 and parameters: {'max_iter': 300, 'solver': 'sag', 'C': 0.

In [ ]:
mlflow.register_model(
    model_uri="runs:/d0210c58afff4737a306a2fbc5f1ff8d/model",
    name="housing-price-predictor",
)